# Collaborative Filtering for Polymarket

so the main issue here is that polymarket events expire pretty quickly (days/weeks) which means we cant just do normal item-based CF because theres no user overlap between old and new events

my idea: use tags instead of events directly. tags like "Politics" or "Crypto" stick around even when specific events dont

In [19]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from implicit.als import AlternatingLeastSquares
from collections import defaultdict
from dotenv import load_dotenv
from sqlalchemy import create_engine

## loading the data

need user trades with timestamps for the train/test split

In [20]:
from utils import load_trade_tags

df = load_trade_tags()

print(f"got {len(df):,} trades")
print(f"users: {df['user_id'].nunique():,}")
print(f"tags: {df['tag_id'].nunique():,}")
df.head()

Loading cached data from 82 parquet files in 'df_trade_tags_cache/'...
got 14,111,477 trades
users: 8,581
tags: 3,721


,user_id,tag_id,tag_label,timestamp,trade_count
0,0xd5039d967e6aafee9b778f2968120cf61fbd3a14,100215,All,2022-11-22 03:59:32,1
1,0xd5039d967e6aafee9b778f2968120cf61fbd3a14,126,Trump,2022-11-22 03:59:32,1
2,0xd5039d967e6aafee9b778f2968120cf61fbd3a14,143,u.s. 2024 republican presidential nomination,2022-11-22 03:59:32,1
3,0xd5039d967e6aafee9b778f2968120cf61fbd3a14,144,Elections,2022-11-22 03:59:32,1
4,0xd5039d967e6aafee9b778f2968120cf61fbd3a14,160,ron desantis,2022-11-22 03:59:32,1


## train test split

doing temporal split - first 80% of each users trades for training, last 20% for testing. this makes more sense than random because we want to predict future behavior

In [21]:
# sort everything by time first
df = df.sort_values(['user_id', 'timestamp'])

# figure out split point for each user
df['idx'] = df.groupby('user_id').cumcount()
df['total'] = df.groupby('user_id')['timestamp'].transform('size')

# 80% cutoff, but make sure theres at least 1 trade in each set
df['cutoff'] = (df['total'] * 0.8).astype(int)
df['cutoff'] = df['cutoff'].clip(lower=1, upper=df['total']-1)

# only users with 2+ trades (otherwise cant split)
df = df[df['total'] >= 2]

train = df[df['idx'] < df['cutoff']].copy()
test = df[df['idx'] >= df['cutoff']].copy()

# cleanup temp columns
for col in ['idx', 'total', 'cutoff']:
    train.drop(col, axis=1, inplace=True)
    test.drop(col, axis=1, inplace=True)

print(f"train: {len(train):,}")
print(f"test: {len(test):,}")

train: 11,285,771
test: 2,825,704


In [22]:
# aggregate to user-tag pairs
train_agg = train.groupby(['user_id', 'tag_id', 'tag_label'])['trade_count'].sum().reset_index()
test_agg = test.groupby(['user_id', 'tag_id', 'tag_label'])['trade_count'].sum().reset_index()

print(f"train pairs: {len(train_agg):,}")
print(f"test pairs: {len(test_agg):,}")
print(f"users in both: {train_agg['user_id'].nunique():,}")

train pairs: 585,126
test pairs: 269,099
users in both: 8,579


## building the matrix

using implicit feedback here - no explicit ratings, just trade counts. the confidence formula compresses big numbers so power users dont dominate everything

confidence = 1 + alpha * log(1 + trades)

tried a few values for alpha, 40 seemed to work ok

In [23]:
# need to map user/tag ids to matrix indices
users = train_agg['user_id'].astype('category')
tags = train_agg['tag_id'].astype('category')

user2idx = {u: i for i, u in enumerate(users.cat.categories)}
idx2user = {i: u for u, i in user2idx.items()}
tag2idx = {t: i for i, t in enumerate(tags.cat.categories)}
idx2tag = {i: t for t, i in tag2idx.items()}

n_users = len(user2idx)
n_tags = len(tag2idx)
print(f"matrix will be {n_users} x {n_tags}")

matrix will be 8579 x 3666


In [24]:
# build sparse matrix
alpha = 40

rows = users.cat.codes.values
cols = tags.cat.codes.values
vals = 1 + alpha * np.log1p(train_agg['trade_count'].values)

mat = sparse.csr_matrix((vals, (rows, cols)), shape=(n_users, n_tags))

# how sparse is it?
density = mat.nnz / (mat.shape[0] * mat.shape[1])
print(f"density: {density:.4f} ({density*100:.2f}%)")
print(f"so {(1-density)*100:.1f}% of cells are empty")

density: 0.0186 (1.86%)
so 98.1% of cells are empty


## training

using ALS from implicit library. tried different params:
- factors: started with 32, bumped to 64 for better results
- regularization: 0.05 to prevent overfitting
- iterations: 30 seems enough, loss plateaus after that

In [25]:
model = AlternatingLeastSquares(
    factors=64,
    regularization=0.05,
    iterations=30,
    random_state=42
)

model.fit(mat, show_progress=True)

# save model
model.save("../models/collaborative_filtering_model.npz")

100%|██████████| 30/30 [00:03<00:00,  8.05it/s]


## evaluation

checking precision@10 and hit rate@10
- precision = what fraction of recommendations are actually relevant
- hit rate = what fraction of users got at least one good rec

In [26]:
# get ground truth - what tags did each user actually interact with in test set
test_tags = test_agg.groupby('user_id')['tag_id'].apply(set).to_dict()

k = 10
precisions = []
hits = 0
total = 0

for uid, true_tags in test_tags.items():
    if uid not in user2idx:
        continue
    
    uidx = user2idx[uid]
    
    # get recommendations
    rec_idx, scores = model.recommend(
        uidx,
        mat[uidx],
        N=k,
        filter_already_liked_items=True
    )
    
    rec_tags = {idx2tag[i] for i in rec_idx if i in idx2tag}
    
    # how many hits?
    n_hits = len(rec_tags & true_tags)
    precisions.append(n_hits / k)
    
    if n_hits > 0:
        hits += 1
    total += 1

print(f"precision@{k}: {np.mean(precisions):.4f}")
print(f"hit rate@{k}: {hits/total:.4f}")
print(f"tested on {total} users")

precision@10: 0.1006
hit rate@10: 0.4985
tested on 8579 users


## recommending actual events

ok so the model gives us tag preferences but we need to recommend events. gonna score each event by averaging the scores of its tags

In [27]:
from utils import load_events_with_tags

events_df = load_events_with_tags()

print(f"loaded {events_df['event_id'].nunique()} events")

Loading events_df from 7 cached files in 'events_tag_cache/'...
loaded 65056 events


In [28]:
# build lookup tables
event_to_tags = defaultdict(list)
event_titles = {}
tag_names = {}

for _, row in events_df.iterrows():
    eid = row['event_id']
    tid = row['tag_id']
    
    if tid in tag2idx:
        tidx = tag2idx[tid]
        event_to_tags[eid].append(tidx)
        tag_names[tidx] = row['tag_label']
    
    event_titles[eid] = row['title']

print(f"{len(event_to_tags)} events have known tags")

65056 events have known tags


In [29]:
def get_event_reccomandations(user_id, n=5):    
    if user_id not in user2idx:
        return None
    
    uidx = user2idx[user_id]
    
    # get this users tag preferences
    user_vec = model.user_factors[uidx]
    tag_scores = model.item_factors.dot(user_vec)
    
    # score events by their tags
    scores = []
    for eid, tidxs in event_to_tags.items():
        s = np.mean([tag_scores[t] for t in tidxs])
        scores.append((eid, s))
    
    scores.sort(key=lambda x: -x[1])
    
    results = []
    for eid, score in scores[:n]:
        tags = set([tag_names[t] for t in event_to_tags[eid]])
        results.append({
            'title': event_titles[eid],
            'score': score,
            'tags': tags
        })
    
    return results

In [30]:
# test
sample_user = "0x000d257d2dc7616feaef4ae0f14600fdf50a758e£"#list(user2idx.keys())[10]
# sample_user = "0x000d257d2dc7616feaef4ae0f14600fdf50a758e"
recs = get_event_reccomandations(sample_user, n=5)

print(f"recs for {sample_user}:\n")
for i, r in enumerate(recs, 1):
    print(f"{i}. {r['title']}")
    print(f"   tags: {r['tags']}")
    print(f"   score: {r['score']:.3f}")
    print()

recs for 0x000d257d2dc7616feaef4ae0f14600fdf50a758e£:



TypeError: 'NoneType' object is not iterable